# ***Cross-Validation***

**Before we tune XGBoost**, we need to answer:

> Is `R² = 0.8394` genuinely good, or did we just get a favorable test split?

That's where **cross-validation** comes in.

Instead of evaluating on one split:

```text
Train ───────────────── Test
       model →          R²
```

we divide the training data into multiple folds:

```text
Fold 1: [TEST] [TRAIN] [TRAIN] [TRAIN] [TRAIN]
Fold 2: [TRAIN] [TEST] [TRAIN] [TRAIN] [TRAIN]
Fold 3: [TRAIN] [TRAIN] [TEST] [TRAIN] [TRAIN]
Fold 4: [TRAIN] [TRAIN] [TRAIN] [TEST] [TRAIN]
Fold 5: [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST]
```

Then calculate the score on each fold and average them.

This gives us a much more reliable estimate.

---

## 5-Fold CV for XGBoost

Run:

```python
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=kfold,
    scoring="r2",
    n_jobs=-1
)

print("XGBoost Cross-Validation R²")
print("--------------------------")
print("Fold scores:", cv_scores)
print("Mean R²    :", cv_scores.mean())
print("Std R²     :", cv_scores.std())
```

### What we're looking for

For example:

```text
Fold scores:
[0.82, 0.85, 0.84, 0.81, 0.86]

Mean R² = 0.836
Std     = 0.018
```

That would tell us the model is **consistently strong**, rather than succeeding only on our particular test split.



# ***Hyperparameter Tuning***

Now we're going to optimize XGBoost.

Instead of guessing:

```python
max_depth=3
learning_rate=0.03
```

we'll systematically test combinations.

The main parameters we'll tune are:

```text
learning_rate
max_depth
n_estimators
subsample
colsample_bytree
min_child_weight
```

### Important

We're going to use **RandomizedSearchCV**, not GridSearchCV initially.

Why?

Suppose we test:

```text
5 learning rates
4 max depths
5 n_estimators
3 subsamples
3 column samples
```

That's:

```text
5 × 4 × 5 × 3 × 3 = 900 combinations
```

and each combination involves cross-validation.

That's unnecessarily expensive.

Randomized search samples a fixed number of combinations.

---

## Set up tuning

Run:

```python id="c0j8vl"
from sklearn.model_selection import RandomizedSearchCV
```

Create parameter space:

```python id="s6v2gc"
param_distributions = {
    "model__n_estimators": [300, 500, 700, 1000, 1500],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}
```

Notice the prefix:

```text
model__
```

That's because our XGBoost model is inside:

```python
Pipeline([
    ("preprocessor", ...),
    ("model", XGBRegressor(...))
])
```

So sklearn accesses its parameters through:

```text
model__parameter
```

---

## Randomized Search

```python id="k5y8f2"
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="r2",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
```

### Why `n_iter=30`?

We're testing **30 randomly selected parameter combinations** rather than hundreds/thousands.

With:

```text
30 combinations × 5 folds
```

we train approximately:

```text
150 models
```

That's enough for our first search without going crazy computationally.

---

## Run tuning

```python id="b4qj3k"
random_search.fit(X_train, y_train)
```

This may take some time depending on your CPU.

Then:

```python id="w3x9x1"
print("Best CV R²:", random_search.best_score_)
print("\nBest Parameters:")
print(random_search.best_params_)
```
